# P0-6 — Answer evaluation (pairwise LLM-as-judge)

**Graded deliverable:** compare ≥ 2 answer variants on a held set and document the winner.

The RAG answer generator ([`rag_helper.py`](../serve/rag_helper.py)) has one swappable knob — the
coaching **instructions** — with two variants, everything else (retrieval, grounding,
citations) held identical so the comparison isolates that one change:

- **BLUNT** — 2–3 concrete actions, imperative, no explanation.
- **WHY** — the same actions, each with the short reason it works, so the player learns the
  principle, not just the instruction.

**Method** ([`eval_answer.py`](eval_answer.py)): for every held-out question, generate both
answers and have a Haiku **judge** pick the better *coaching* answer. Pairwise (not the
course's reference-based judge) because coaching questions have **no single correct answer** —
there is nothing to score against; a judge only has to say *which of two is better*, which LLMs
do reliably. **Position bias is removed** by `compare`: each pair is judged in **both orders**
and a win is credited only if it survives the swap — order-dependent disagreement collapses to
a tie.

This notebook presents the **committed result**; it does not re-run the eval. The full census
is `eval_answer.main()` — 2,836 live calls (1,418 generations + 1,418 judgements), ~4.7 hr at
the throttled 10 RPM. Its tally is saved to `data/eval/p0-6_pairwise.json`.

In [1]:
import json, math
from pathlib import Path

p = Path("../../data/eval/p0-6_pairwise.json")
if not p.exists():
    p = Path("data/eval/p0-6_pairwise.json")
res = json.loads(p.read_text())

c = res["counts"]
print(f"N questions      : {res['n_questions']}  ({res['sampling']})")
print(f"backend / model  : {res['backend']} / {res['model']}")
print(f"calls            : {res['calls']}  ({res['calls_breakdown']['generation']} gen + "
      f"{res['calls_breakdown']['judge']} judge, both orders)")
print()
print("pairwise tally (debiased):")
for k in ("BLUNT", "WHY", "tie"):
    print(f"  {k:<6}{c[k]:>4}")

N questions      : 709  (full ground-truth census (all rows, no sampling))
backend / model  : bedrock / claude-haiku-4-5
calls            : 2836  (1418 gen + 1418 judge, both orders)

pairwise tally (debiased):
  BLUNT  217
  WHY    298
  tie    194


In [2]:
# Is WHY's lead real, or noise? Sign test on the decisive (non-tie) verdicts.
blunt, why, tie = c["BLUNT"], c["WHY"], c["tie"]
decisive = blunt + why
mean = decisive * 0.5
sd = math.sqrt(decisive * 0.25)
z = (why - mean) / sd
p_two_sided = 2 * (1 - 0.5 * (1 + math.erf(abs(z) / math.sqrt(2))))

print(f"decisive verdicts : {decisive}   (ties excluded: {tie}, {tie/res['n_questions']:.1%})")
print(f"WHY share         : {why}/{decisive} = {why/decisive:.1%}")
print(f"sign test         : z = {z:.2f},  two-sided p = {p_two_sided:.2e}")
print(f"verdict           : WHY wins" + (" (significant, p < 0.001)" if p_two_sided < 1e-3
      else ""))

decisive verdicts : 515   (ties excluded: 194, 27.4%)
WHY share         : 298/515 = 57.9%
sign test         : z = 3.57,  two-sided p = 3.58e-04
verdict           : WHY wins (significant, p < 0.001)


## Conclusion

**WHY is the documented winner.** Over the full 709-question census it takes the decisive
verdicts **298–217** (57.9%) — a lead ~3.6 SD off a 50/50 split, **p < 0.001** by a sign test.

This is worth flagging against an earlier read: a first pass at **N = 50** came out
**19–20–11 — a dead heat** — and we noted "chasing N won't fix a gap this small." That was
wrong. The lean toward WHY was **real all along**, just below N = 50's resolution; the census
resolves it cleanly. The ~27% tie rate is expected — the two variants share retrieval,
grounding, and citations and differ only in whether each action carries its "because", so on
many questions they are genuinely equivalent.

**Decision:** ship **WHY**. BLUNT was only ever a foil — a coach
that refuses to explain itself is worse on purpose — and the census confirms the explanatory
version is *measurably* better, not merely no-worse.

*Scope of the claim:* this is a within-eval result — a Haiku judge's preference over synthetic,
situational questions. It establishes that WHY is the better variant **on the criteria the
judge was given** (specific, actionable, direct, useful for game review), which is exactly what
P0-6 asks for; it is not a human-subjects study.